In [0]:
from pyspark.sql import functions as F

silver_folder = 'abfss://silver@asterisktotle01.dfs.core.windows.net/PHHousing'

# 1. Load Silver Delta Table
df = spark.read.format('delta').load(silver_folder + '/dim_fam')

# 2. Process and Transform Pipeline
df_gold = (
    df.drop('geography','avgIncome', 'avgExpenses')
      # Cast to decimal(18,2) to eliminate floating-point trailing precision artifacts
      .withColumn('capacityToPayPagibig', (F.col('monthlyIncome') * 0.35).cast('decimal(18,2)'))
      .withColumn('capacityToPayBank', (F.col('monthlyIncome') * 0.30).cast('decimal(18,2)'))
      .withColumn(
          'reliability',
          F.when(F.col('families') >= 200, 'High')
           .when(F.col('families') >= 100, 'Moderate')
           .when(F.col('families') >= 30, 'Low')
           .otherwise('Very Low')
      )
)

# 3. Display Result


In [0]:
dim_fam = spark.read.format('delta').load(silver_folder + '/dim_fam')
display(dim_fam.select('geography','families'))